Nota de Privacidade: Este notebook demonstra o tratamento dos dados do SIM. Para garantir a privacidade e respeitar a LGPD, os outputs exibidos abaixo utilizam uma amostra fictícia (amostra_ficticia_fc.csv). O código original rodou sobre a base real de 261 óbitos, que não é disponibilizada neste repositório.

In [5]:
import pandas as pd
import numpy as np

In [ ]:
df_ficticio = pd.read_csv("../dados/df_sim_demo.csv", sep=",")
cid_10 = pd.read_csv("../dados/cid_10_categorias.csv", sep=";", encoding="latin1")

display (df_ficticio)
display(cid_10)

,CODMUNRES,DTOBITO,IDADE,SEXO,RACACOR,CAUSABAS,TIPOBITO
0,999999,1012023,457,1,1,X000,2
1,999999,15022023,101,2,2,Y111,1
2,999999,20032023,203,1,4,Z222,2
3,999999,10042023,999,2,9,W333,2
4,999999,5052023,502,1,5,V444,2


,CAT,CLASSIF,DESCRICAO,DESCRABREV,REFER,EXCLUIDOS
0,A00,NaN,Cólera,A00 Colera,NaN,NaN
1,A01,NaN,Febres tifóide e paratifóide,A01 Febres tifoide e paratifoide,NaN,NaN
2,A02,NaN,Outras infecções por Salmonella,A02 Outr infecc p/Salmonella,NaN,NaN
3,A03,NaN,Shiguelose,A03 Shiguelose,NaN,NaN
4,A04,NaN,Outras infecções intestinais bacterianas,A04 Outr infecc intestinais bacter,NaN,NaN
...,...,...,...,...,...,...
2040,U80,NaN,Agente resistente à penicilina e antibióticos ...,U80 Agente resist penicilina e antibiót relac,NaN,NaN
2041,U81,NaN,Agente resistente à vancomicina e antibióticos...,U81 Agente resist vancomicina e antibiót relac,NaN,NaN
2042,U88,NaN,Agente resistente a múltiplos antibióticos,U88 Agente resistente a múltiplos antibióticos,NaN,NaN
2043,U89,NaN,Agente resistente a outros antibióticos e a an...,U89 Agente resist outr antibiót e antibiót NE,NaN,NaN


In [7]:
display (df_ficticio.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   CODMUNRES  5 non-null      int64 
 1   DTOBITO    5 non-null      int64 
 2   IDADE      5 non-null      int64 
 3   SEXO       5 non-null      int64 
 4   RACACOR    5 non-null      int64 
 5   CAUSABAS   5 non-null      object
 6   TIPOBITO   5 non-null      int64 
dtypes: int64(6), object(1)
memory usage: 412.0+ bytes


None

In [8]:
df_ficticio = df_ficticio.drop(columns = "CODMUNRES")

df_ficticio["DTOBITO"] = df_ficticio["DTOBITO"].astype(str).str.zfill(8)
df_ficticio["DTOBITO"] = pd.to_datetime(df_ficticio["DTOBITO"], format = "%d%m%Y")
df_ficticio["SEXO"] = df_ficticio["SEXO"].map({1 : "Masculino", 2 : "Feminino"})
df_ficticio["TIPOBITO"] = df_ficticio["TIPOBITO"].map({1 : "Fetal", 2 : "Não Fetal"})

# Função para conversão do código de raça/cor

def escrever_raca(raca):
  if raca == 1:
    return "Branca"
  elif raca == 2:
    return "Preta"
  elif raca == 3:
    return "Amarela"
  elif raca == 4:
    return "Parda"
  elif raca == 5:
    return "Indígena"
  else:
    return "Ignorado/Não preenchido"


df_ficticio["RACACOR"] = df_ficticio["RACACOR"].apply(escrever_raca)

# Classificação por idade

MAPA_UNIDADES_IDADE = {
    "1": "Minutos",
    "2": "Horas",
    "3": "Meses",
    "4": "Anos",
}

def classificar_idade(idade):
    if pd.isna(idade) or len(str(idade)) != 3:
        return pd.NA, "Não preenchido/suspeito"
    idade_str = str(idade)
    if idade_str[0] in MAPA_UNIDADES_IDADE:
        return int(idade_str[1:]), MAPA_UNIDADES_IDADE[idade_str[0]]
    elif idade_str[0] == "5":
        return 100 + int(idade_str[1:]), "Centenário"
    elif idade_str[0] == "9":
        return pd.NA, "Ignorado"
    else:
        return pd.NA, "Não preenchido/suspeito"

df_ficticio[["IDADE", "CLASSE_IDADE"]] = pd.DataFrame(df_ficticio["IDADE"].apply(classificar_idade).tolist(), index = df_ficticio.index)


def criar_faixa_etaria(df, col_idade="IDADE", col_classe="CLASSE_IDADE"):
    
    faixa = pd.Series("Não preenchido/Ignorado", index=df.index)
    
   
    faixa[df[col_classe].isin(['Minutos', 'Horas', 'Meses'])] = 'Menor de 1 ano'
    
    is_anos = df[col_classe].isin(['Anos', 'Centenário'])
    
    faixa[is_anos & (df[col_idade] <= 9)] = 'Criança'
    faixa[is_anos & (df[col_idade].between(10, 19))] = 'Adolescente'
    faixa[is_anos & (df[col_idade].between(20, 59))] = 'Adulto'
    faixa[is_anos & (df[col_idade] >= 60)] = 'Idoso'
    
    return faixa

df_ficticio["FAIXAETARIA"] = criar_faixa_etaria(df_ficticio)


# Unindo as bases 
df_ficticio["CID3"] = df_ficticio["CAUSABAS"].str[:3]
df_ficticio = df_ficticio.merge(
    cid_10[["CAT", "DESCRICAO"]], 
    left_on = "CID3",
    right_on= "CAT",
    how = "left"
)

display(df_ficticio)



,DTOBITO,IDADE,SEXO,RACACOR,CAUSABAS,TIPOBITO,CLASSE_IDADE,FAIXAETARIA,CID3,CAT,DESCRICAO
0,2023-01-01,57,Masculino,Branca,X000,Não Fetal,Anos,Adulto,X00,X00,Exposição a fogo não-controlado em um edifício...
1,2023-02-15,1,Feminino,Preta,Y111,Fetal,Minutos,Menor de 1 ano,Y11,Y11,Envenenamento [intoxicação] por e exposição a ...
2,2023-03-20,3,Masculino,Parda,Z222,Não Fetal,Horas,Menor de 1 ano,Z22,Z22,Portador de doença infecciosa
3,2023-04-10,<NA>,Feminino,Ignorado/Não preenchido,W333,Não Fetal,Ignorado,Não preenchido/Ignorado,W33,W33,"Rifle, espingarda e armas de fogo de maior tam..."
4,2023-05-05,102,Masculino,Indígena,V444,Não Fetal,Centenário,Idoso,V44,V44,Ocupante de um automóvel [carro] traumatizado ...


In [9]:

df_ficticio = df_ficticio.drop(columns = "CID3")

df_ficticio = df_ficticio[["DTOBITO", "IDADE", "CLASSE_IDADE","FAIXAETARIA", "SEXO", "RACACOR", "CAUSABAS", "CAT", "DESCRICAO", "TIPOBITO"]]

display(df_ficticio)

,DTOBITO,IDADE,CLASSE_IDADE,FAIXAETARIA,SEXO,RACACOR,CAUSABAS,CAT,DESCRICAO,TIPOBITO
0,2023-01-01,57,Anos,Adulto,Masculino,Branca,X000,X00,Exposição a fogo não-controlado em um edifício...,Não Fetal
1,2023-02-15,1,Minutos,Menor de 1 ano,Feminino,Preta,Y111,Y11,Envenenamento [intoxicação] por e exposição a ...,Fetal
2,2023-03-20,3,Horas,Menor de 1 ano,Masculino,Parda,Z222,Z22,Portador de doença infecciosa,Não Fetal
3,2023-04-10,<NA>,Ignorado,Não preenchido/Ignorado,Feminino,Ignorado/Não preenchido,W333,W33,"Rifle, espingarda e armas de fogo de maior tam...",Não Fetal
4,2023-05-05,102,Centenário,Idoso,Masculino,Indígena,V444,V44,Ocupante de um automóvel [carro] traumatizado ...,Não Fetal


In [ ]:
df_ficticio.to_csv("../dados/df_ficticio_trabalhado.csv", index=False)